# 🧠 U-Net Medical Image Segmentation
### Brain MRI Tumor Segmentation — LGG Kaggle Dataset

**Dataset:** [mateuszbuda/lgg-mri-segmentation](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation)  
**Model:** U-Net (Ronneberger et al., MICCAI 2015)  
**Task:** Binary pixel-wise segmentation of brain tumors in MRI scans  
**Metrics:** Dice Coefficient, IoU, Pixel Accuracy

---
**Notebook Sections:**
1. Environment Setup
2. Dataset Exploration & Visualisation
3. Dataset Class & DataLoaders
4. U-Net Architecture
5. Training Loop
6. Evaluation on Test Set
7. Visualise Predictions
8. Save Model
9. Results Summary

## 1. Environment Setup

In [ ]:
# Install required libraries (run once on Kaggle / Colab)
!pip install albumentations --quiet
!pip install torchsummary --quiet

In [ ]:
import os
import glob
import random
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torch.optim.lr_scheduler import ReduceLROnPlateau

import albumentations as A
from albumentations.pytorch import ToTensorV2

print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU              : {torch.cuda.get_device_name(0)}')

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nUsing device     : {DEVICE}')

## 2. Configuration

In [ ]:
# ════════════════════════════════════════════
#  All hyperparameters in one place
# ════════════════════════════════════════════

# Kaggle input path (adjust if running locally)
KAGGLE_ROOT  = '/kaggle/input/lgg-mri-segmentation/kaggle_3m'
# If running locally, change to:
# KAGGLE_ROOT = 'ai/data/kaggle_3m'

MODEL_SAVE_PATH = 'unet_model.pt'

IMAGE_HEIGHT   = 256
IMAGE_WIDTH    = 256

TRAIN_SPLIT    = 0.80
VAL_SPLIT      = 0.10
# TEST = remaining 0.10

BATCH_SIZE     = 16
NUM_EPOCHS     = 50
LEARNING_RATE  = 1e-4
WEIGHT_DECAY   = 1e-5
PATIENCE       = 10          # early stopping
MASK_THRESHOLD = 0.5
FEATURES       = [64, 128, 256, 512]
DROPOUT        = 0.3
BCE_WEIGHT     = 0.5
DICE_WEIGHT    = 0.5

print('Configuration loaded ✓')

## 3. Dataset Exploration & Visualisation

In [ ]:
# Scan all image / mask pairs
all_files   = sorted(glob.glob(os.path.join(KAGGLE_ROOT, '**', '*.tif'), recursive=True))
image_paths = [f for f in all_files if '_mask' not in f]
mask_paths  = [f.replace('.tif', '_mask.tif') for f in image_paths]

# Validate pairs
valid = [(img, msk) for img, msk in zip(image_paths, mask_paths) if os.path.exists(msk)]
image_paths, mask_paths = zip(*valid)
image_paths, mask_paths = list(image_paths), list(mask_paths)

print(f'Total image-mask pairs : {len(image_paths)}')

# Count patients
patients = set(os.path.basename(os.path.dirname(p)) for p in image_paths)
print(f'Total patients         : {len(patients)}')

# Count positive masks (tumour present)
positive = 0
for msk_path in mask_paths:
    msk = np.array(Image.open(msk_path).convert('L'))
    if msk.max() > 0:
        positive += 1

print(f'Slices WITH tumour     : {positive}  ({100*positive/len(mask_paths):.1f}%)')
print(f'Slices WITHOUT tumour  : {len(mask_paths) - positive}')

In [ ]:
# Visualise 8 random positive samples
positive_pairs = [
    (img, msk) for img, msk in zip(image_paths, mask_paths)
    if np.array(Image.open(msk).convert('L')).max() > 0
]
samples = random.sample(positive_pairs, 8)

fig, axes = plt.subplots(8, 3, figsize=(12, 28))
col_labels = ['MRI Slice', 'Ground Truth Mask', 'Overlay']
for col, label in enumerate(col_labels):
    axes[0, col].set_title(label, fontsize=13, fontweight='bold')

for row, (img_path, msk_path) in enumerate(samples):
    img = np.array(Image.open(img_path).convert('RGB'), dtype=np.uint8)
    msk = np.array(Image.open(msk_path).convert('L'), dtype=np.float32)
    msk = (msk > 0).astype(np.uint8)

    # Overlay
    overlay = img.copy()
    overlay[msk == 1] = (
        0.6 * overlay[msk == 1] + 0.4 * np.array([255, 50, 50])
    ).astype(np.uint8)

    axes[row, 0].imshow(img)
    axes[row, 1].imshow(msk, cmap='gray')
    axes[row, 2].imshow(overlay)
    patient = os.path.basename(os.path.dirname(img_path))
    axes[row, 0].set_ylabel(patient[:20], fontsize=7)
    for ax in axes[row]: ax.axis('off')

plt.suptitle('Sample Brain MRI Slices with Tumour Masks', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('dataset_samples.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved dataset_samples.png')

## 4. Dataset Class & DataLoaders

In [ ]:
# ── Augmentation transforms ────────────────────────────────────

train_transform = A.Compose([
    A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1,
                       rotate_limit=30, p=0.5),
    A.ElasticTransform(p=0.3),
    A.GridDistortion(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.2,
                               contrast_limit=0.2, p=0.4),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std =(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std =(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


# ── Dataset class ──────────────────────────────────────────────

class BrainMRIDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert('RGB'),
                         dtype=np.uint8)
        mask  = np.array(Image.open(self.mask_paths[idx]).convert('L'),
                         dtype=np.float32)
        mask  = (mask > 0).astype(np.float32)

        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug['image']          # [3, H, W]
            mask  = aug['mask']           # [H, W]

        return image, mask.unsqueeze(0)   # mask → [1, H, W]


# ── Train / Val / Test split ───────────────────────────────────

total   = len(image_paths)
indices = np.random.default_rng(SEED).permutation(total).tolist()

n_train = int(total * TRAIN_SPLIT)
n_val   = int(total * VAL_SPLIT)

train_idx = indices[:n_train]
val_idx   = indices[n_train : n_train + n_val]
test_idx  = indices[n_train + n_val :]

def make_subset(idx_list, transform):
    imgs  = [image_paths[i] for i in idx_list]
    masks = [mask_paths[i]  for i in idx_list]
    return BrainMRIDataset(imgs, masks, transform)

train_ds = make_subset(train_idx, train_transform)
val_ds   = make_subset(val_idx,   val_transform)
test_ds  = make_subset(test_idx,  val_transform)

print(f'Train : {len(train_ds)} samples')
print(f'Val   : {len(val_ds)}   samples')
print(f'Test  : {len(test_ds)}  samples')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=1,
                          shuffle=False, num_workers=2)

# Sanity check shapes
imgs, masks = next(iter(train_loader))
print(f'\nImage batch : {imgs.shape}')   # [16, 3, 256, 256]
print(f'Mask  batch : {masks.shape}')   # [16, 1, 256, 256]

## 5. U-Net Architecture

In [ ]:
class DoubleConv(nn.Module):
    """Conv → BN → ReLU → Conv → BN → ReLU"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class EncoderBlock(nn.Module):
    """DoubleConv + MaxPool → returns (skip, downsampled)"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = DoubleConv(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2, 2)
    def forward(self, x):
        skip = self.conv(x)
        return skip, self.pool(skip)


class DecoderBlock(nn.Module):
    """ConvTranspose2d upsample → concat skip → DoubleConv"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = DoubleConv(out_ch * 2, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = TF.resize(x, skip.shape[2:])
        return self.conv(torch.cat([skip, x], dim=1))


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1,
                 features=None, dropout=DROPOUT):
        super().__init__()
        if features is None:
            features = FEATURES

        self.encoders = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.encoders.append(EncoderBlock(ch, f))
            ch = f

        self.bottleneck = nn.Sequential(
            DoubleConv(features[-1], features[-1] * 2),
            nn.Dropout2d(p=dropout),
        )

        self.decoders = nn.ModuleList()
        bt_ch = features[-1] * 2
        for f in reversed(features):
            self.decoders.append(DecoderBlock(bt_ch, f))
            bt_ch = f

        self.out_conv = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for enc in self.encoders:
            skip, x = enc(x)
            skips.append(skip)
        x = self.bottleneck(x)
        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)
        return self.out_conv(x)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = UNet().to(DEVICE)
print(f'Trainable parameters: {model.count_params():,}')

# Quick shape test
with torch.no_grad():
    dummy = torch.randn(2, 3, IMAGE_HEIGHT, IMAGE_WIDTH).to(DEVICE)
    out   = model(dummy)
    print(f'Input  → {dummy.shape}')
    print(f'Output → {out.shape}')    # should be [2, 1, 256, 256]

## 6. Loss Functions & Metrics

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(-1)
        tgts  = targets.view(-1)
        inter = (probs * tgts).sum()
        return 1 - (2*inter + self.smooth) / (probs.sum() + tgts.sum() + self.smooth)

class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce  = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
    def forward(self, logits, targets):
        return BCE_WEIGHT * self.bce(logits, targets) + \
               DICE_WEIGHT * self.dice(logits, targets)

def dice_coeff(pred, target, smooth=1.0):
    p, t = pred.flatten(), target.flatten()
    return float((2*(p*t).sum() + smooth) / (p.sum() + t.sum() + smooth))

def iou_score(pred, target, smooth=1.0):
    p, t = pred.flatten(), target.flatten()
    inter = (p*t).sum()
    return float((inter + smooth) / (p.sum() + t.sum() - inter + smooth))

criterion = CombinedLoss()
optimizer = optim.Adam(model.parameters(),
                       lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode='max',
                              factor=0.5, patience=5, verbose=True)
print('Loss, metrics, optimiser ready ✓')

## 7. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_dice = 0.0, 0.0
    for images, masks in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, masks)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        with torch.no_grad():
            preds = (torch.sigmoid(logits) > MASK_THRESHOLD).float()
            total_dice += dice_coeff(preds.cpu().numpy(), masks.cpu().numpy())
        total_loss += loss.item()
    n = len(loader)
    return total_loss / n, total_dice / n


@torch.no_grad()
def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_dice, total_iou = 0.0, 0.0, 0.0
    for images, masks in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        logits = model(images)
        loss   = criterion(logits, masks)
        preds  = (torch.sigmoid(logits) > MASK_THRESHOLD).float()
        p_np, m_np = preds.cpu().numpy(), masks.cpu().numpy()
        total_loss += loss.item()
        total_dice += dice_coeff(p_np, m_np)
        total_iou  += iou_score(p_np, m_np)
    n = len(loader)
    return total_loss / n, total_dice / n, total_iou / n


# ── Main training loop ─────────────────────────────────────────
history = {'train_loss':[], 'val_loss':[], 'train_dice':[], 'val_dice':[]}
best_dice, patience_counter = 0.0, 0

print(f'Starting training for {NUM_EPOCHS} epochs on {DEVICE}\n')
print(f'{"Epoch":<8} {"T-Loss":<10} {"T-Dice":<10} {"V-Loss":<10} {"V-Dice":<10} {"V-IoU":<10} {"Time"}')
print('─' * 68)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_dice         = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_dice, vl_iou = val_epoch(model, val_loader, criterion)

    scheduler.step(vl_dice)
    elapsed = time.time() - t0

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_dice'].append(tr_dice)
    history['val_dice'].append(vl_dice)

    marker = ''
    if vl_dice > best_dice:
        best_dice = vl_dice
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        marker = '  ← best'
    else:
        patience_counter += 1

    print(f'{epoch:<8} {tr_loss:<10.4f} {tr_dice:<10.4f} '
          f'{vl_loss:<10.4f} {vl_dice:<10.4f} {vl_iou:<10.4f} '
          f'{elapsed:.1f}s{marker}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
        break

print(f'\nTraining complete. Best Val Dice = {best_dice:.4f}')
print(f'Model saved → {MODEL_SAVE_PATH}')

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history['train_loss']) + 1)

axes[0].plot(ep, history['train_loss'], label='Train Loss', color='steelblue', lw=2)
axes[0].plot(ep, history['val_loss'],   label='Val Loss',   color='coral',     lw=2)
axes[0].set_title('Loss Curve', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history['train_dice'], label='Train Dice', color='steelblue', lw=2)
axes[1].plot(ep, history['val_dice'],   label='Val Dice',   color='coral',     lw=2)
axes[1].axhline(0.80, color='green', linestyle='--', lw=1.2, label='Target (0.80)')
axes[1].set_title('Dice Score Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Dice')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('U-Net Training History', fontsize=14)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

## 9. Evaluation on Test Set

In [ ]:
# Load best model weights
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
model.eval()

all_dice, all_iou, all_acc = [], [], []

with torch.no_grad():
    for images, masks in tqdm(test_loader, desc='Evaluating'):
        images = images.to(DEVICE)
        logits = model(images)
        preds  = (torch.sigmoid(logits) > MASK_THRESHOLD).float()
        p_np   = preds.cpu().numpy().flatten()
        m_np   = masks.numpy().flatten()

        tp = (p_np * m_np).sum()
        fp = (p_np * (1 - m_np)).sum()
        fn = ((1 - p_np) * m_np).sum()
        tn = ((1 - p_np) * (1 - m_np)).sum()

        all_dice.append((2*tp + 1e-6) / (2*tp + fp + fn + 1e-6))
        all_iou.append( (tp + 1e-6)   / (tp + fp + fn + 1e-6))
        all_acc.append( (tp + tn)     / (tp + tn + fp + fn + 1e-6))

print('\n' + '='*45)
print('  Test Set Results')
print('='*45)
print(f'  Dice Coefficient : {np.mean(all_dice):.4f} ± {np.std(all_dice):.4f}')
print(f'  IoU (Jaccard)    : {np.mean(all_iou):.4f} ± {np.std(all_iou):.4f}')
print(f'  Pixel Accuracy   : {np.mean(all_acc):.4f} ± {np.std(all_acc):.4f}')
print('='*45)

## 10. Visualise Test Predictions

In [ ]:
def denorm(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    t = tensor.permute(1,2,0).numpy()
    t = t * std + mean
    return np.clip(t * 255, 0, 255).astype(np.uint8)

# Collect 6 test samples
sample_images, sample_masks, sample_preds = [], [], []
with torch.no_grad():
    for imgs, msks in test_loader:
        logits = model(imgs.to(DEVICE))
        preds  = (torch.sigmoid(logits) > MASK_THRESHOLD).float().cpu()
        sample_images.append(imgs[0]);  sample_masks.append(msks[0])
        sample_preds.append(preds[0])
        if len(sample_images) == 6: break

fig, axes = plt.subplots(6, 4, figsize=(16, 24))
titles = ['MRI Slice', 'Ground Truth', 'Prediction', 'Overlay']
for col, t in enumerate(titles):
    axes[0, col].set_title(t, fontsize=12, fontweight='bold')

for row in range(6):
    img  = denorm(sample_images[row])
    gt   = sample_masks[row][0].numpy()
    pred = sample_preds[row][0].numpy()
    ovl  = img.copy()
    ovl[pred == 1] = (0.55*ovl[pred==1] + 0.45*np.array([255,50,50])).astype(np.uint8)

    axes[row,0].imshow(img)
    axes[row,1].imshow(gt,   cmap='gray')
    axes[row,2].imshow(pred, cmap='gray')
    axes[row,3].imshow(ovl)

    d = (2*np.sum(pred*gt)+1e-6)/(np.sum(pred)+np.sum(gt)+1e-6)
    axes[row,3].set_xlabel(f'Dice={d:.3f}', fontsize=9)
    for ax in axes[row]: ax.axis('off')

plt.suptitle('Test Set Predictions', fontsize=14)
plt.tight_layout()
plt.savefig('test_predictions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved test_predictions.png')

## 11. Results Summary

In [ ]:
print('\n' + '█'*50)
print('  FINAL RESULTS SUMMARY')
print('█'*50)
print(f'  Dataset        : LGG Brain MRI Segmentation')
print(f'  Total samples  : {len(image_paths)}')
print(f'  Image size     : {IMAGE_HEIGHT} × {IMAGE_WIDTH}')
print(f'  Batch size     : {BATCH_SIZE}')
print(f'  Epochs trained : {len(history["train_loss"])}')
print(f'  Best Val Dice  : {best_dice:.4f}')
print('─'*50)
print(f'  TEST SET METRICS')
print(f'  Dice           : {np.mean(all_dice):.4f}')
print(f'  IoU            : {np.mean(all_iou):.4f}')
print(f'  Pixel Accuracy : {np.mean(all_acc):.4f}')
print('─'*50)
print(f'  Model saved to : {MODEL_SAVE_PATH}')
print('█'*50)

In [ ]:
# Download model from Kaggle notebook output
# The file 'unet_model.pt' will appear in the output panel on the right
# Click the file → Download to save it locally for the Flask backend
print(f'File size: {os.path.getsize(MODEL_SAVE_PATH) / 1e6:.1f} MB')
print('Download unet_model.pt from the Kaggle output panel ↗')